# 🧠💀 Brain Rot Studio
Make your own **songs** 🎵 and **GIFs** 🎞️ with code.

**How to use this notebook**
1. Run every cell marked ⚙️ **ENGINE** once, top to bottom (press ▶ or `Shift+Enter`). Don't edit them.
2. Edit the cells marked ✏️ **YOUR TURN**, then run them again.
3. Change something, re-run, listen/look. Repeat!

*Using JupyterLite (in the browser)? The first run is slow while it downloads numpy and Pillow, and making a GIF can take 10 to 30 seconds. That's normal. Files you save show up in the file list on the left (right-click, Download).*

In [ ]:
# ⚙️ ENGINE: setup (run once)
import os, io, sys, time, math, wave, shutil, subprocess
import numpy as np
from IPython.display import Audio, Image as IPImage, Video, display
from PIL import Image, ImageDraw, ImageFont, ImageEnhance
print("Ready! 🔥")

# Part 1: 🎵 Songs
A song is built from **tracks**. Each track = one sound + a pattern of steps.

```
kick   x . . . x . . . x . . . x . . x
snare  . . . . x . . . . . . . x . . .
         ^ one step (a 16th note)       x = play it    . = silence
```

In [ ]:
# ⚙️ ENGINE: the sound library (run once)
SR = 22050
SOUNDS = {}

def _reg(name, pitched=False):
    def deco(f):
        SOUNDS[name] = (f, pitched)
        return f
    return deco

def _t(d): return np.arange(int(SR * d)) / SR
def _lp(x, k): return np.convolve(x, np.ones(k) / k, mode="same")
def _phase(f): return 2 * np.pi * np.cumsum(f) / SR
def _saw(ph): return 2 * ((ph / (2 * np.pi)) % 1) - 1

@_reg("kick")
def _kick(freq=None):
    t = _t(.4); return np.sin(_phase(45 + 130 * np.exp(-t * 28))) * np.exp(-t * 8)

@_reg("snare")
def _snare(freq=None):
    t = _t(.25); n = np.random.default_rng(1).standard_normal(len(t))
    return n * np.exp(-t * 22) * .6 + np.sin(2 * np.pi * 190 * t) * np.exp(-t * 30) * .5

@_reg("hat")
def _hat(freq=None):
    t = _t(.08); n = np.diff(np.random.default_rng(2).standard_normal(len(t)), prepend=0)
    return n * np.exp(-t * 60) * .5

@_reg("clap")
def _clap(freq=None):
    t = _t(.3); n = np.random.default_rng(4).standard_normal(len(t)); env = np.zeros(len(t))
    for d in (0, .015, .03):
        m = t >= d; env[m] += np.exp(-(t[m] - d) * 60)
    return n * env * np.exp(-t * 8) * .5

@_reg("boom")      # the classic vine-boom
def _boom(freq=None):
    t = _t(1.6); x = np.sin(_phase(32 + 60 * np.exp(-t * 4))) * np.exp(-t * 2.2)
    return np.tanh(3 * x)

@_reg("airhorn")
def _airhorn(freq=None):
    t = _t(.8); out = 0
    for f0 in (466, 587, 698):
        out = out + _saw(_phase(f0 * (1 + .01 * np.sin(2 * np.pi * 6 * t))))
    env = np.minimum(1, t / .02) * np.clip((.8 - t) / .15, 0, 1)
    return _lp(out, 3) * env * .4

@_reg("bruh")
def _bruh(freq=None):
    t = _t(.55); x = _lp(_saw(_phase(150 * np.exp(-t * .9))), 25)
    return x * np.minimum(1, t / .03) * np.exp(-t * 2.5) * 2

@_reg("zap")
def _zap(freq=None):
    t = _t(.3); return np.sign(np.sin(_phase(150 + 2500 * np.exp(-t * 9)))) * np.exp(-t * 8) * .4

@_reg("oof")
def _oof(freq=None):
    t = _t(.3); f = 380 - 700 * t
    return (np.sin(_phase(f)) + .5 * np.sin(_phase(2 * f))) * np.exp(-t * 8) * .6

@_reg("riser")
def _riser(freq=None):
    t = _t(1.0); n = np.random.default_rng(5).standard_normal(len(t))
    return (np.sin(_phase(200 + 1800 * t ** 2)) * .5 + n * .2) * t

@_reg("glitch")
def _glitch(freq=None):
    rng = np.random.default_rng(3); out = np.zeros(int(SR * .3)); pos = 0
    while pos < len(out):
        L = int(SR * rng.uniform(.01, .04)); f = rng.choice([300, 600, 900, 1500, 2400])
        seg = np.sign(np.sin(2 * np.pi * f * np.arange(L) / SR)) * .3
        for _ in range(int(rng.integers(1, 3))):
            e = min(len(out), pos + L); out[pos:e] = seg[:e - pos]; pos = e
            if pos >= len(out): break
    return out

# --- pitched sounds: use note names like C4, F#3, Bb5 ---
@_reg("bell", True)
def _bell(freq=None):
    f0 = freq or 880; t = _t(1.2)
    return sum(a * np.sin(2 * np.pi * f0 * m * t) * np.exp(-t * d)
               for m, a, d in [(1, 1, 3), (2.76, .5, 5), (5.4, .25, 8)]) * .5

@_reg("bass", True)
def _bass(freq=None):
    f0 = freq or 55; t = _t(.35); return np.tanh(2.5 * np.sin(2 * np.pi * f0 * t)) * np.exp(-t * 6)

@_reg("lead", True)
def _lead(freq=None):
    f0 = freq or 440; t = _t(.3); return _lp(np.sign(np.sin(2 * np.pi * f0 * t)), 2) * np.exp(-t * 7) * .35

NOTE_NAMES = {"C": 0, "D": 2, "E": 4, "F": 5, "G": 7, "A": 9, "B": 11}
def note_freq(n):
    semi = NOTE_NAMES[n[0].upper()]; rest = n[1:]
    if rest[:1] == "#": semi += 1; rest = rest[1:]
    elif rest[:1] == "b": semi -= 1; rest = rest[1:]
    midi = 12 * (int(rest) + 1) + semi
    return 440 * 2 ** ((midi - 69) / 12)

print("Sounds loaded:", ", ".join(SOUNDS))

In [ ]:
# ⚙️ ENGINE: the sequencer (run once)
def _tokens(p):
    p = p.strip()
    return p.split() if " " in p else list(p)

def make_song(tracks, bpm=140, loops=2, volume=0.9):
    parsed = []
    for tr in tracks:
        name, pattern = tr[0], tr[1]
        vol = tr[2] if len(tr) > 2 else 1.0
        if name not in SOUNDS:
            raise ValueError(f"No sound called '{name}'. Pick from: {', '.join(SOUNDS)}")
        parsed.append((name, _tokens(pattern), vol))
    steps = max(len(t) for _, t, _ in parsed)
    step = int(SR * 60 / bpm / 4)
    mix = np.zeros(step * steps * loops + int(SR * 1.8))
    for name, toks, vol in parsed:
        fn, pitched = SOUNDS[name]
        for lp in range(loops):
            for i in range(steps):
                tok = toks[i % len(toks)]
                if tok in (".", "-"): continue
                freq = None
                if pitched and tok.lower() != "x":
                    try: freq = note_freq(tok)
                    except Exception: raise ValueError(f"'{tok}' isn't a note. Use things like C4, F#3, Bb5, or . for silence")
                snd = fn(freq) * vol
                start = (lp * steps + i) * step
                end = min(len(mix), start + len(snd))
                mix[start:end] += snd[:end - start]
    peak = np.abs(mix).max()
    return mix / peak * volume if peak > 0 else mix

def play_song(song):
    display(Audio(song, rate=SR))

def save_song(song, filename="mysong.wav"):
    with wave.open(filename, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR)
        w.writeframes((np.clip(song, -1, 1) * 32767).astype(np.int16).tobytes())
    print("Saved", filename)

def preview_sounds():
    for name, (fn, pitched) in SOUNDS.items():
        print(("🎹 " if pitched else "🥁 ") + name + ("   (pitched: use notes like C4)" if pitched else ""))
        display(Audio(fn(), rate=SR))

preview_sounds()

In [ ]:
# ✏️ YOUR TURN: build a song
#   x = play the sound    . = rest
#   Pitched sounds (bell, bass, lead) use note names. Put spaces between notes.
#   Every pattern should be 16 steps (or 8, 4... it repeats to fill the bar).

BPM = 140

song = make_song([
    ("kick",  "x...x...x...x..x"),
    ("snare", "....x.......x..."),
    ("hat",   "x.x.x.x.x.x.x.xx"),
    ("boom",  "x..............."),
    ("bell",  "E5 . . . G5 . . . A5 . . . G5 . E5 ."),
    ("bass",  "C2 . . . C2 . . . F2 . . . G2 . . ."),
], bpm=BPM, loops=4)

play_song(song)
# save_song(song, "mysong.wav")   # remove the # to save it

### 🎯 Song challenges
- Change `BPM` to 100, then 180. What happens?
- Replace `boom` with `airhorn` or `bruh`. Put it on a different step.
- Add a new track: `("glitch", "...........x....")`
- Make a 3rd bell melody that's *worse* on purpose. Use notes like `C4 D4 E4`.
- Add a 3rd number to a track to change its volume: `("hat", "x.x.x.x.", 0.4)`

# Part 2: 🕺 Brr Brr Patapim dance GIF
Pick a list of **moves**. Patapim does them in order, one move after another, on the beat.

```
moves=["bounce", "stomp", "wiggle", "spin", "shake"]
```

In [ ]:
# ⚙️ ENGINE: Patapim and the dance maker (run once)
from math import sin, cos, pi, exp

BROWN, DARK, TAN, NOSE, GREEN, BLUE = (139, 90, 43), (60, 35, 15), (214, 160, 110), (205, 125, 95), (70, 170, 70), (40, 90, 220)

def draw_patapim(arm_l=15, arm_r=15, leg_l=3, leg_r=3, nose=1.0, k=2):
    W = 256 * k
    im = Image.new("RGBA", (W, W), (0, 0, 0, 0)); d = ImageDraw.Draw(im)
    box = lambda b: [v * k for v in b]
    def limb(x, y, ang, length, side, width, color):
        a = math.radians(ang); ex, ey = x + side * length * math.sin(a), y + length * math.cos(a)
        d.line([x * k, y * k, ex * k, ey * k], fill=color, width=int(width * k))
        for cx, cy in ((x, y), (ex, ey)):
            d.ellipse(box([cx - width / 2, cy - width / 2, cx + width / 2, cy + width / 2]), fill=color)
        return ex, ey
    # legs + big shoes
    for x, ang, side in ((110, leg_l, -1), (146, leg_r, 1)):
        ex, ey = limb(x, 190, ang, 42, side, 18, BROWN)
        d.ellipse(box([ex - 24 + side * 8, ey - 8, ex + 24 + side * 8, ey + 16]), fill=BLUE, outline=DARK, width=2 * k)
    # trunk body
    d.rounded_rectangle(box([86, 118, 170, 196]), radius=18 * k, fill=BROWN, outline=DARK, width=3 * k)
    for x in (104, 128, 150): d.line(box([x, 135, x, 185]), fill=DARK, width=2 * k)
    # hair, head
    for b in ([68, 18, 128, 62], [98, 8, 158, 54], [126, 18, 188, 64], [84, 30, 172, 60]):
        d.ellipse(box(b), fill=GREEN, outline=(40, 110, 40), width=2 * k)
    d.ellipse(box([76, 42, 180, 134]), fill=TAN, outline=DARK, width=3 * k)
    # arms + hands
    for x, ang, side in ((90, arm_l, -1), (166, arm_r, 1)):
        ex, ey = limb(x, 138, ang, 52, side, 14, BROWN)
        d.ellipse(box([ex - 10, ey - 10, ex + 10, ey + 10]), fill=TAN, outline=DARK, width=2 * k)
    # face
    for cx in (113, 143):
        d.ellipse(box([cx - 14, 70, cx + 14, 98]), fill="white", outline=DARK, width=2 * k)
        d.ellipse(box([cx - 4, 79, cx + 8, 93]), fill="black")
    d.arc(box([106, 112, 150, 134]), 20, 160, fill=DARK, width=3 * k)
    nw, nh = 36 * nose, 44 * nose
    d.ellipse(box([128 - nw / 2, 100 - nh / 2, 128 + nw / 2, 100 + nh / 2]), fill=NOSE, outline=DARK, width=2 * k)
    for dx in (-6, 6):
        d.ellipse(box([128 + dx * nose - 3, 108 * 1 + (nose - 1) * 10 - 3, 128 + dx * nose + 3, 108 + (nose - 1) * 10 + 3]), fill=DARK)
    return im.resize((256, 256), Image.BOX)

_pat_cache = {}
def get_patapim(arm_l, arm_r, leg_l, leg_r, nose):
    key = (round(arm_l / 3), round(arm_r / 3), round(leg_l / 3), round(leg_r / 3), round(nose * 20))
    if key not in _pat_cache:
        _pat_cache[key] = draw_patapim(key[0] * 3, key[1] * 3, key[2] * 3, key[3] * 3, key[4] / 20)
    return _pat_cache[key]

# ---------- the moves ----------
MOVES = {}
def move(name, desc):
    def deco(f): MOVES[name] = (desc, f); return f
    return deco

DEFAULT = dict(dx=0, dy=0, rot=0, sx=1, sy=1, sc=1, arm_l=15, arm_r=15, leg_l=3, leg_r=3, nose=1)

@move("bounce", "jumps up and down, squashing on landing")
def _bounce(t, u, rng):
    s = abs(sin(pi * t)); return dict(dy=-.12 * s, sy=.88 + .12 * s, sx=1.1 - .1 * s, arm_l=15 + 70 * s, arm_r=15 + 70 * s)

@move("stomp", "stomps one foot, then the other")
def _stomp(t, u, rng):
    a = sin(pi * t); return dict(leg_l=45 * max(a, 0), leg_r=45 * max(-a, 0), dy=-.03 * abs(a), arm_l=40, arm_r=40)

@move("spin", "spins all the way around")
def _spin(t, u, rng): return dict(rot=360 * u, arm_l=90, arm_r=90)

@move("wiggle", "wiggles side to side")
def _wiggle(t, u, rng):
    s = sin(4 * pi * t); return dict(rot=14 * s, arm_l=60 + 50 * s, arm_r=60 - 50 * s)

@move("wave", "waves both arms")
def _wave(t, u, rng):
    s = sin(2 * pi * t); return dict(arm_l=110 + 70 * s, arm_r=110 - 70 * s)

@move("nose", "the nose grows and shrinks (brrr)")
def _nose(t, u, rng): return dict(nose=1 + .7 * abs(sin(pi * t)), rot=6 * sin(2 * pi * t))

@move("shake", "vibrates like BRRR BRRR")
def _shake(t, u, rng):
    return dict(dx=rng.uniform(-.03, .03), dy=rng.uniform(-.02, .02), rot=rng.uniform(-5, 5), nose=1.15, arm_l=30, arm_r=30)

@move("zoom", "zooms in and out")
def _zoom(t, u, rng): return dict(sc=.75 + .6 * sin(pi * u))

@move("flip", "turns around like a coin")
def _flip(t, u, rng): return dict(sx=max(abs(cos(2 * pi * u)), .05), arm_l=60, arm_r=60)

@move("kick", "fast alternating kicks")
def _kick(t, u, rng):
    s = sin(4 * pi * t); return dict(leg_l=60 * max(s, 0), leg_r=60 * max(-s, 0), arm_l=90, arm_r=90)

@move("sway", "slides left and right")
def _sway(t, u, rng):
    s = sin(pi * t); return dict(dx=.12 * s, rot=-8 * s, arm_l=45 + 30 * s, arm_r=45 - 30 * s)

# ---------- backgrounds, effects, text ----------
def _font(size):
    for p in ["DejaVuSans-Bold.ttf", "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", "arialbd.ttf", "Arial Bold.ttf"]:
        try: return ImageFont.truetype(p, size)
        except OSError: pass
    try: return ImageFont.load_default(size)
    except TypeError: return ImageFont.load_default()

def text_image(text, color="white", outline="black"):
    f = _font(120); probe = ImageDraw.Draw(Image.new("RGBA", (10, 10)))
    l, t, r, b = probe.multiline_textbbox((0, 0), text, font=f, stroke_width=8)
    im = Image.new("RGBA", (r - l + 24, b - t + 24), (0, 0, 0, 0))
    ImageDraw.Draw(im).multiline_text((12 - l, 12 - t), text, font=f, fill=color, stroke_width=8, stroke_fill=outline, align="center")
    return im

BACKGROUNDS = ["tunnel", "rainbow", "strobe", "static", "black"]
POST = ["flash", "glitch", "deepfry", "shake_cam"]

def _hsv(h, s, v):
    i = np.floor(h * 6).astype(int) % 6; f = h * 6 - np.floor(h * 6)
    v = np.full_like(h, v); p = v * (1 - s); q = v * (1 - f * s); t = v * (1 - (1 - f) * s)
    ch = lambda a: np.choose(i, a)
    return (np.stack([ch([v, q, p, p, t, v]), ch([t, v, v, q, p, p]), ch([p, p, t, v, v, q])], -1) * 255).astype(np.uint8)

def _geometry(W):
    Y, X = np.mgrid[0:W, 0:W].astype(np.float32) / W
    return dict(X=X, Y=Y, R=np.hypot(X - .5, Y - .5) * 30)

def _background(bg, W, loop, beat, rng, geo):
    X, Y = geo["X"], geo["Y"]
    if bg == "rainbow": arr = _hsv(((X + Y) / 2 + loop) % 1, .9, 1.0)
    elif bg == "strobe":
        pal = [(255, 0, 200), (0, 255, 255), (255, 255, 0), (20, 20, 20)]
        arr = np.full((W, W, 3), pal[int(beat * 2) % 4], dtype=np.uint8)
    elif bg == "static": arr = np.repeat(rng.integers(0, 256, (W, W, 1)), 3, axis=2).astype(np.uint8)
    elif bg == "tunnel":
        arr = np.where((np.sin(geo["R"] - 2 * np.pi * beat) > 0)[..., None], (150, 0, 255), (10, 0, 30)).astype(np.uint8)
    elif bg == "black": arr = np.zeros((W, W, 3), dtype=np.uint8)
    elif isinstance(bg, str) and os.path.exists(bg): arr = np.array(Image.open(bg).convert("RGB").resize((W, W)))
    else: raise ValueError(f"Unknown background '{bg}'. Pick from: {', '.join(BACKGROUNDS)} (or an image file name)")
    return Image.fromarray(arr)

def _post(frame, effects, beat, rng):
    for e in effects:
        if e == "deepfry":
            frame = ImageEnhance.Color(frame).enhance(3); frame = ImageEnhance.Contrast(frame).enhance(1.8)
            buf = io.BytesIO(); frame.save(buf, "JPEG", quality=12); frame = Image.open(buf).convert("RGB")
        elif e == "glitch":
            a = np.array(frame); W = a.shape[0]
            for _ in range(3):
                y0 = int(rng.integers(0, W - 20)); h = int(rng.integers(5, 25))
                a[y0:y0 + h] = np.roll(a[y0:y0 + h], int(rng.integers(-40, 40)), axis=1)
            frame = Image.fromarray(a)
        elif e == "flash": frame = ImageEnhance.Brightness(frame).enhance(1 + 1.2 * exp(-8 * (beat % 1)))
        elif e == "shake_cam":
            frame = frame.transform(frame.size, Image.AFFINE, (1, 0, int(rng.integers(-8, 9)), 0, 1, int(rng.integers(-8, 9))))
        else: raise ValueError(f"Unknown effect '{e}'. Pick from: {', '.join(POST)}")
    return frame

# ---------- the dance maker ----------
def make_dance(moves, beats_per_move=2, bpm=140, fps=15, size=240, background="tunnel",
               caption="", character="patapim", post=(), filename="patapim.gif"):
    bad = [m for m in moves if m not in MOVES]
    if bad: raise ValueError(f"Unknown move {bad}. Pick from: {', '.join(MOVES)}")
    custom = None
    if character != "patapim":
        if not os.path.exists(character): raise ValueError(f"Can't find the image file '{character}'")
        custom = Image.open(character).convert("RGBA")
    cap = text_image(caption) if caption else None
    geo = _geometry(size); t0 = time.time(); Q = getattr(Image, "Quantize", Image)
    per = max(2, round(beats_per_move * 60 / bpm * fps)); nf = per * len(moves)
    total_beats = beats_per_move * len(moves); frames = []
    print(f"Making {nf} frames...", flush=True)
    for f in range(nf):
        name = moves[f // per]; u = (f % per) / per; t = u * beats_per_move
        loop = f / nf; beat = loop * total_beats; rng = np.random.default_rng(f)
        p = dict(DEFAULT); p.update(MOVES[name][1](t, u, rng))
        frame = _background(background, size, loop, beat, rng, geo)
        img = custom if custom is not None else get_patapim(p["arm_l"], p["arm_r"], p["leg_l"], p["leg_r"], p["nose"])
        base = .6 * size * p["sc"]
        w = max(2, int(base * p["sx"])); h = max(2, int(base * p["sy"] * img.height / img.width))
        im = img.resize((w, h), Image.BILINEAR).rotate(p["rot"], expand=True, resample=Image.BILINEAR)
        frame.paste(im, (int((.5 + p["dx"]) * size - im.width / 2), int((.54 + p["dy"]) * size - im.height / 2)), im)
        if cap is not None:
            cw = int(.85 * size * (1 + .1 * exp(-6 * (beat % 1)))); ch = max(2, int(cw * cap.height / cap.width))
            c = cap.resize((cw, ch), Image.BILINEAR); frame.paste(c, (int(size / 2 - cw / 2), int(.04 * size)), c)
        frames.append(_post(frame, post, beat, rng).quantize(colors=64, method=Q.FASTOCTREE))
        if f % 10 == 0: print(f"  frame {f + 1}/{nf}", flush=True)
    print("  saving...", flush=True)
    frames[0].save(filename, save_all=True, append_images=frames[1:], duration=int(1000 / fps), loop=0, disposal=2)
    print(f"Saved {filename}  ({nf} frames, {os.path.getsize(filename) // 1024} KB, {time.time() - t0:.1f} seconds)")
    display(IPImage(data=open(filename, "rb").read(), format="gif"))

print("Dance engine loaded 🕺")

In [ ]:
# 👀 Meet Patapim and see all the moves you can use
display(draw_patapim().resize((200, 200)))
for name, (desc, _) in MOVES.items():
    print(f"{name:8} {desc}")
print("\nBackgrounds:", ", ".join(BACKGROUNDS))
print("Extra effects (post):", ", ".join(POST))

In [ ]:
# ⚡ Quick test: makes a tiny GIF. If this works, everything works.
make_dance(["bounce"], beats_per_move=2, size=160, fps=10, caption="TEST", filename="test.gif")

In [ ]:
# ✏️ YOUR TURN: choreograph the dance
#   moves = the order of the moves. Each one lasts beats_per_move beats.

make_dance(
    moves=["bounce", "stomp", "wiggle", "spin", "shake"],
    beats_per_move=2,
    bpm=140,
    background="tunnel",
    caption="BRR BRR PATAPIM",
    post=["flash"],
)

### 🎯 Dance challenges
- Make the dance **longer**: add more moves. Repeat the same one twice if you like it.
- Change `beats_per_move` to 1 for fast moves, or 4 for slow ones.
- Try other backgrounds: `"rainbow"`, `"strobe"`, `"static"`.
- Add `post=["deepfry", "glitch"]` for maximum chaos.
- Change the `caption` to anything you want.
- Use **your own character**: upload a picture, then `character="myface.png"`. (Whole-body moves only: bounce, spin, wiggle, shake, zoom, flip, sway.)

# Part 3: 🎬 Put them together (optional)
Your dance loops in time with your song when both use the same `BPM`. A good match: 2 beats per move × 4 or 8 moves.
This cell needs **ffmpeg** (already on Google Colab).

In [ ]:
# ✏️ YOUR TURN: song + GIF = video
def make_video(song, gif="patapim.gif", filename="brainrot.mp4"):
    if sys.platform == "emscripten":
        print("Video export doesn't work in the browser (JupyterLite). Use your GIF and song files instead:")
        print("right-click them in the file list on the left and choose Download."); return
    if not shutil.which("ffmpeg"):
        print("ffmpeg not found. On Colab it's built in; on your own computer, install ffmpeg first."); return
    save_song(song, "_song.wav")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-stream_loop", "-1", "-i", gif, "-i", "_song.wav",
                    "-shortest", "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2", "-pix_fmt", "yuv420p", filename], check=True)
    display(Video(filename, embed=True, width=320))

make_video(song)